<a href="https://colab.research.google.com/github/iav2002/Assignment_Advanced_Topics_In_DeepLearning/blob/main/Part3_DQN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 3: DQN on Warehouse Navigation

We train a Deep Q-Network agent to navigate a 5x5 warehouse grid and reach a pallet, using the visual embeddings learned in Part 2 as part of the state representation.

The investigation has two threads.  First we compare three DQN variants in an additive fashion:
1. A vanilla version,
2. Experience replay,
3. Before + Target network on top.

Afterwards we run an ablation on the reward function we designed, since reward shaping is where the real design comes to play.

We start from a baseline Reward function provitional and then we wil improve it along our investigation, as well as improving the complexity of the gridd

In [11]:
import sys, os, json, random, time
from pathlib import Path
from collections import deque

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [2]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdvancedDL')
EMB_PATH = DRIVE_ROOT / 'embeddings' / 'mean_embeddings.npy'
ENV_PATH = DRIVE_ROOT / 'WarehouseEnv.py'
RESULTS_ROOT = DRIVE_ROOT / 'results_part3'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

assert EMB_PATH.exists(), f"missing: {EMB_PATH}"
assert ENV_PATH.exists(), f"missing: {ENV_PATH}"
print("paths ok")

Mounted at /content/drive
paths ok


In [12]:
# add Drive folder to path so the warehouse file can be imported
sys.path.insert(0, str(DRIVE_ROOT))

# clear any cached version because in case of overwriting
if 'WarehouseEnv' in sys.modules:
    del sys.modules['WarehouseEnv']

import gymnasium
from WarehouseEnv import WarehouseEnv
print("env imported ok")

env imported ok


## Loading the Part 2 embeddings

The mean embeddings come from Part 2's best ResNet50 model (Variant C of Experiment 1, 99.05% test accuracy). They are stored as a dict keyed by class name, each pointing to a 2048-dim vector.

The environment looks them up by integer ID (0=floor, 1=wall, 2=pallet, 3=sign), so we remap the keys before passing the dict in.

In [14]:
emb_str = np.load(EMB_PATH, allow_pickle=True).item()
print("classes:", list(emb_str.keys()))
print("vector dim:", emb_str['floor'].shape)

# remap str keys to int IDs the env expects
NAME_TO_ID = {'floor': 0, 'wall': 1, 'pallet': 2, 'sign': 3}
emb = {NAME_TO_ID[k]: v.astype(np.float32) for k, v in emb_str.items()}
print("remapped keys:", list(emb.keys()))

classes: ['floor', 'wall', 'pallet', 'sign']
vector dim: (2048,)
remapped keys: [0, 1, 2, 3]


## Grid layout

We start with a minimal but non trivial 5x5 grid: the agent spawns at (0,0), the pallet sits at (4,4), and a single wall at the center forces the agent to learn a detour rather than going straight diagonal. No signs in this first version. (will change with commits)

When the experiments turn out too easy or the embeddings end up not contributing meaningfully, we'll come back here to make it harder.

Coordinate convention: `grid_map[row, col]` for the matrix, but the agent's position is stored as `(x, y)` where x is column and y is row, matching the env's movement code (LLMs detection).

In [15]:
# 0=floor, 1=wall, 2=pallet, 3=sign
GRID = [
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 0, 0, 2],
]

# quick visual check
sym = {0:'.', 1:'#', 2:'P', 3:'S'}
for row in GRID:
    print(' '.join(sym[c] for c in row))

. . . . .
. . . . .
. . # . .
. . . . .
. . . . P


In [16]:
#SMOKE TEST
env = WarehouseEnv(grid_map=GRID, embeddings_dict=emb)

print("action space:", env.action_space)
print("obs space shape:", env.observation_space.shape)
print("target coords:", env.target_coords)

obs, info = env.reset()
print("initial obs shape:", obs.shape)
print("first 5 dims (xy + emb start):", obs[:5])

# also test one step to confirm the reward function works
obs, reward, term, trunc, info = env.step(3)  # action 3 = right
print(f"after step right: pos={env.agent_pos}, reward={reward}, term={term}")

action space: Discrete(4)
obs space shape: (2050,)
target coords: [4 4]
initial obs shape: (2050,)
first 5 dims (xy + emb start): [0.         0.         1.1220449  0.06864746 0.02758316]
after step right: pos=[1 0], reward=-0.05, term=False


## Random baseline

Before training anything, we measure how a random agent performs. This is just a sanity check that the env runs end-to-end across many episodes, and also a reference for future variations (the real ones).

We define three per episode metrics that we'll reuse in along the notebook: success (1 if the pallet was reached, 0 otherwise), episode length in steps, and total accumulated reward. Episodes are short to make them fast

Our "easy" grid takes 8 steps to solve, and we are giving 100 MAx steps (expected to have a high accuracy)

In [30]:
MAX_STEPS = 100  # cap per episode, to solve it takes 8

def run_random(env, n_episodes=100, seed=0):
    rng = np.random.default_rng(seed)
    successes, lengths, rewards = [], [], []

    for ep in range(n_episodes):
        env.reset()
        ep_reward = 0.0
        for t in range(MAX_STEPS):
            action = rng.integers(0, env.action_space.n)
            _, r, term, trunc, _ = env.step(int(action))
            ep_reward += r
            if term or trunc:
                break
        successes.append(int(term))
        lengths.append(t + 1)
        rewards.append(ep_reward)

    return np.array(successes), np.array(lengths), np.array(rewards)

# run it
succ, lens, rews = run_random(env, n_episodes=100, seed=0)

print(f"Success rate:   {succ.mean():.2%}")
print(f"Avg ep length:  {lens.mean():.1f} steps")
print(f"Avg ep reward:  {rews.mean():.2f}")
print(f"Episodes that reached the pallet: {succ.sum()}/100")

Success rate:   66.00%
Avg ep length:  69.0 steps
Avg ep reward:  0.86
Episodes that reached the pallet: 66/100
